In [6]:
import pandas as pd
df = pd.read_csv(r"C:\Users\HP\Downloads\hospital_operations_dataset.csv")
print(df.info())


<class 'pandas.DataFrame'>
RangeIndex: 5075 entries, 0 to 5074
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Patient_ID                  5075 non-null   str    
 1   Visit_Date                  5075 non-null   str    
 2   Age                         5075 non-null   int64  
 3   Gender                      4972 non-null   str    
 4   Disease                     5075 non-null   str    
 5   Severity                    5075 non-null   str    
 6   Admission_Type              5075 non-null   str    
 7   Length_of_Stay              5025 non-null   float64
 8   Bed_Required                5075 non-null   str    
 9   ICU_Required                5075 non-null   str    
 10  Oxygen_Required             5075 non-null   str    
 11  Ventilator_Required         5075 non-null   str    
 12  Total_Beds_Available        5075 non-null   int64  
 13  Beds_Occupied               5075 non-null   

In [7]:
df

,Patient_ID,Visit_Date,Age,Gender,Disease,Severity,Admission_Type,Length_of_Stay,Bed_Required,ICU_Required,...,Total_Beds_Available,Beds_Occupied,ICU_Beds_Available,ICU_Beds_Occupied,Oxygen_Units_Used,Month,Season,Day_of_Week,Department,Readmission_Within_30_Days
0,P00624,2023-03-21,63,Male,Food Poisoning,Mild,OPD,0.0,No,No,...,247,156,19,9,3.0,3,Spring,Tuesday,Gastroenterology,No
1,P02862,2023-03-04,44,Male,Liver Disease,Severe,Emergency,9.0,No,No,...,145,92,28,13,4.0,3,Spring,Saturday,Gastroenterology,No
2,P01271,2024-11-10,52,Female,Anemia,Moderate,Inpatient,2.0,Yes,No,...,105,62,20,9,1.0,11,Autumn,Sunday,General Medicine,Yes
3,P02681,2023-02-27,63,Male,Thyroid Disorder,Moderate,OPD,0.0,No,No,...,203,136,21,13,1.0,2,Winter,Monday,General Medicine,No
4,P04980,2023-07-09,72,Male,Heart Disease,Severe,Emergency,11.0,Yes,Yes,...,194,118,31,12,2.0,7,Monsoon,Sunday,ICU,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5070,P02596,2023-06-10,6,Male,Tuberculosis,Severe,Inpatient,7.0,Yes,Yes,...,146,80,28,11,184.0,6,Summer,Saturday,ICU,No
5071,P01958,2023-05-27,72,Female,Anemia,Severe,Inpatient,7.0,No,No,...,250,160,12,8,64.0,5,Summer,Saturday,General Medicine,No
5072,P02468,2023-08-17,42,Female,Arthritis,Mild,Emergency,1.0,Yes,No,...,125,82,11,6,3.0,8,Monsoon,Thursday,Orthopedics,No
5073,P00746,2024-10-02,89,Female,Pneumonia,Moderate,Inpatient,7.0,Yes,No,...,132,74,33,17,35.0,10,Autumn,Wednesday,Pulmonology,No


In [10]:

import numpy as np

# 2. Remove exact duplicate rows
df = df.drop_duplicates().copy()

# 3. Clean whitespace and standardize text casing
string_cols = df.select_dtypes(include="object").columns
for col in string_cols:
    df[col] = df[col].str.strip()

# Standardize specific categorical columns
title_case_cols = [
    "Gender", "Disease", "Severity", "Admission_Type", 
    "Bed_Required", "ICU_Required", "Oxygen_Required", 
    "Ventilator_Required", "Season", "Day_of_Week", 
    "Readmission_Within_30_Days"
]
for col in title_case_cols:
    df[col] = df[col].str.title()

# Standardize Department while keeping "ICU" and "OPD" uppercase
df["Department"] = df["Department"].str.title().replace({"Icu": "ICU"})
df["Admission_Type"] = df["Admission_Type"].replace({"Opd": "OPD"})

# 4. Standardize Visit_Date and fix derived date columns
df["Visit_Date"] = pd.to_datetime(df["Visit_Date"], format="mixed", dayfirst=False)
df["Month"] = df["Visit_Date"].dt.month
df["Day_of_Week"] = df["Visit_Date"].dt.day_name()

# 5. Handle Outliers and Logical Inconsistencies
# Cap or nullify unrealistic ages (> 100)
df.loc[df["Age"] > 100, "Age"] = np.nan

# Fix Length_of_Stay: OPD visits should be 0; extreme values (>= 90) set to NaN
df.loc[df["Admission_Type"] == "OPD", "Length_of_Stay"] = 0
df.loc[df["Length_of_Stay"] >= 90, "Length_of_Stay"] = np.nan

# Fix Oxygen_Units_Used: If Oxygen_Required is 'No', units should be 0 (or capped at normal baseline)
df.loc[(df["Oxygen_Required"] == "No") & (df["Oxygen_Units_Used"] > 10), "Oxygen_Units_Used"] = 0

# 6. Impute Missing Values
# Fill missing Gender with 'Unknown'
df["Gender"] = df["Gender"].fillna("Unknown")

# Fill missing Department based on the most common department for that Disease
df["Department"] = df["Department"].fillna(
    df.groupby("Disease")["Department"].transform(lambda x: x.mode()[0] if not x.mode().empty else "General Medicine")
)

# Fill missing Length_of_Stay using median grouped by Admission_Type and Severity
df["Length_of_Stay"] = df["Length_of_Stay"].fillna(
    df.groupby(["Admission_Type", "Severity"])["Length_of_Stay"].transform("median")
)

# Fill missing Oxygen_Units_Used using median grouped by Oxygen_Required and Severity
df["Oxygen_Units_Used"] = df["Oxygen_Units_Used"].fillna(
    df.groupby(["Oxygen_Required", "Severity"])["Oxygen_Units_Used"].transform("median")
)

# Fill any remaining missing Age with overall median
df["Age"] = df["Age"].fillna(df["Age"].median())

# 7. Enforce final data types
int_cols = [
    "Age", "Length_of_Stay", "Total_Beds_Available", 
    "Beds_Occupied", "ICU_Beds_Available", "ICU_Beds_Occupied", 
    "Oxygen_Units_Used", "Month"
]
for col in int_cols:
    df[col] = df[col].round().astype(int)




C:\Users\HP\AppData\Local\Temp\ipykernel_25264\1019789604.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_cols = df.select_dtypes(include="object").columns


In [11]:
df

,Patient_ID,Visit_Date,Age,Gender,Disease,Severity,Admission_Type,Length_of_Stay,Bed_Required,ICU_Required,...,Total_Beds_Available,Beds_Occupied,ICU_Beds_Available,ICU_Beds_Occupied,Oxygen_Units_Used,Month,Season,Day_of_Week,Department,Readmission_Within_30_Days
0,P00624,2023-03-21,63,Male,Food Poisoning,Mild,OPD,0,No,No,...,247,156,19,9,3,3,Spring,Tuesday,Gastroenterology,No
1,P02862,2023-03-04,44,Male,Liver Disease,Severe,Emergency,9,No,No,...,145,92,28,13,4,3,Spring,Saturday,Gastroenterology,No
2,P01271,2024-11-10,52,Female,Anemia,Moderate,Inpatient,2,Yes,No,...,105,62,20,9,1,11,Autumn,Sunday,General Medicine,Yes
3,P02681,2023-02-27,63,Male,Thyroid Disorder,Moderate,OPD,0,No,No,...,203,136,21,13,1,2,Winter,Monday,General Medicine,No
4,P04980,2023-07-09,72,Male,Heart Disease,Severe,Emergency,11,Yes,Yes,...,194,118,31,12,2,7,Monsoon,Sunday,ICU,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5070,P02596,2023-06-10,6,Male,Tuberculosis,Severe,Inpatient,7,Yes,Yes,...,146,80,28,11,184,6,Summer,Saturday,ICU,No
5071,P01958,2023-05-27,72,Female,Anemia,Severe,Inpatient,7,No,No,...,250,160,12,8,64,5,Summer,Saturday,General Medicine,No
5072,P02468,2023-08-17,42,Female,Arthritis,Mild,Emergency,1,Yes,No,...,125,82,11,6,3,8,Monsoon,Thursday,Orthopedics,No
5073,P00746,2024-10-02,89,Female,Pneumonia,Moderate,Inpatient,7,Yes,No,...,132,74,33,17,35,10,Autumn,Wednesday,Pulmonology,No


In [12]:
pip install openyxl

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement openyxl (from versions: none)
ERROR: No matching distribution found for openyxl


In [14]:
df.to_excel("cleaned_data.xlsx",index=False)